# CAMELYON17: Validation Center vs. Independent Test Center

Experiments E2 (CAMELYON17 part) and E4 (`docs/experiments/cross_domain_experiments_details.md`):
Tissue-Region-Net (TRN-Ref configuration) for lymph-node metastasis detection, trained on whole centers and
evaluated on two **unseen** centers. CAMELYON17 has **no in-domain test set**.

| | Split | Centers | Slides (with graph) | Tumor / normal | Use |
|---|---|---|---|---|---|
| Training | `train` | 0, 1, 2 | 207 (203) | 1 : 1 | model fitting |
| Validation | `val` | 3 | 80 (79) | 40 / 40 (balanced) | checkpoint selection and decision threshold |
| Test | `test` | 4 | 100 (98) | 39 / 61 (natural) | independent test (prediction only) |

**How to read the results**

* **Gap = AUC(Val) − AUC(Test)** (as in the docs and the thesis); positive = worse on the test center.
  The validation center also selected the checkpoint and the threshold, so its values are optimistically
  biased: the Gap contains this selection optimism in addition to the center shift. The test center is the
  only unbiased estimate.
* **Only prevalence-robust metrics are compared between centers**: AUC (threshold-free), sensitivity and
  specificity (per class) and, for the multiclass task, OvO macro AUROC, macro-F1 and quadratic κ. Metrics that
  depend on the tumor prevalence (accuracy, PPV, NPV, F1) differ between the centers already because the
  prevalence differs (50 % vs. 38 %); they are only listed in the result table, per center.
* **Validation = point estimates.** Per-slide predictions are stored for the test center only. Validation
  metrics come from `validation_thresholds/validation_thresholds.json` of the same evaluation (binary
  confusion counts reconstructed exactly from the stored rates) and, for multiclass κ, from the training
  history at the selected epoch. Figures that need per-slide scores (ROC, score distributions, oracle
  threshold, metastasis size) use the test center only.
* **Statistics.** 95 % percentile CIs from a patient-level cluster bootstrap of the test center
  (B = `N_BOOT`, 20 patients, 5 lymph nodes each), so they are wide. The CI of the Gap covers the test-center
  sampling only. Model comparisons on the test center are paired on the same resampled patients
  (two-sided bootstrap *p*, Holm-corrected).
* **Runs with a caveat** (configuration `CAVEATS`) are used everywhere but marked with **†**; every figure and
  table that contains them carries the reason as a footnote, and the training curves show it.

**Experiments and the evidence in this notebook**

| Experiment | Question | Evidence |
|---|---|---|
| E2 binary | Does the encoder (Prov-GigaPath vs. ResNet-50) change detection and its transfer to a new center? | Figures 2–5, paired comparisons (slide and region level) |
| E2 multiclass | Same question for negative / ITC / micro / macro | Figure 6 |
| E4 | Does graph-level augmentation (C-aug vs. C-noaug) change the transfer to a new center? | Figures 2–5 and the paired comparisons (test center 4); Figure 9: all five test centers (center rotation) |

**Figures**

1. Training curves with the selected checkpoint
2. AUC on both centers, Gap, ROC on the test center
3. Sensitivity and specificity at the validation threshold (slide counts)
4. Threshold transfer: test-center score distributions and the cost of the transferred threshold
5. Tumor localization on the test center: region and patch level (precision–recall, AUC, AP)
6. Multiclass: AUROC, macro-F1, κ and the test-center confusion matrix
7. E4 over five test centers: AUC, Gap and operating point per test center

Detection by metastasis size (ITC / micro / macro) is given as a table: with 5–6 slides per size class a plot
would suggest more precision than the data have.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from sklearn.metrics import cohen_kappa_score, f1_score, roc_auc_score, roc_curve

REPO_ROOT = Path("/home/stud/hweber/master_code")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.plot_style import IEEE_COLORS, apply_ieee_style, figsize

SPLITS_DIR = REPO_ROOT / "data/splits/camelyon/seed_42"
OUT_DIR = REPO_ROOT / "notebooks/results/camelyon"
OUT_DIR.mkdir(parents=True, exist_ok=True)
for old in OUT_DIR.glob("cam_*"):          # outputs of earlier notebook versions must not linger
    old.unlink()

## Configuration

In [ ]:
# Binary runs: {model: {"eval": evaluation folder, "history": training history, "meta": training meta}}
# The evaluation folder holds split_test/ (per-slide test predictions) and validation_thresholds/ (validation
# metrics of the same checkpoint). Paths are relative to the repository root or absolute; None = not trained /
# not evaluated yet.
RUNS = {
    "Prov-GigaPath": {
        "eval": "results/evaluations/e2-cam-bin-gigapath_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_20261024_142805",
        "history": "results/models/training_history_e2-cam-bin-gigapath_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_segmentation.json",
        "meta": "results/metas/training_metas/e2-cam-bin-gigapath_task-lymph_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_dataset-camelyon_model_type-gnn_ar_78d582ac_mil_cached_training_method-function_20261025_093558.yaml"
    },
    "ResNet-50": {
        "eval": "results/evaluations/e2-cam-bin-resnet50_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_resnet50_gnn_tissue_region_net_graph_resnet50_20261024_142906",
        "history": "results/models/training_history_e2-cam-bin-resnet50_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_resnet50_gnn_tissue_region_net_graph_resnet50_segmentation.json",
        "meta": "results/metas/training_metas/e2-cam-bin-resnet50_task-lymph_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_resnet50_dataset-camelyon_model_ty_7471c69f_mil_cached_training_method-function_20261025_070625.yaml"
    },
    "Prov-GigaPath · C-noaug": {
        "eval": "results/evaluations/e4-cam-bin-noaug_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_20261026_093106",
        "history": "results/models/training_history_e4-cam-bin-noaug_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_segmentation.json",
        "meta": "results/metas/training_metas/e4-cam-bin-noaug_task-lymph_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_dataset-camelyon_model_type-gnn_archi_ed3f28c9_mil_cached_training_method-function_20261027_030813.yaml"
    }
}
# Multiclass runs (4 classes), same keys.
RUNS_MC = {
    "Prov-GigaPath": {
        "eval": "results/evaluations/e2-cam-multi-gigapath_lymph_dataset-camelyon_cfg-model_config_tissue_region_camelyon_subtype_cls_gnn_tissue_region_net_graph_pov-gigapath_20261025_204748",
        "history": "results/models/training_history_e2-cam-multi-gigapath_lymph_dataset-camelyon_cfg-model_config_tissue_region_camelyon_subtype_cls_gnn_tissue_region_net_graph_pov-gigapath.json",
        "meta": "results/metas/training_metas/e2-cam-multi-gigapath_task-lymph_cfg-model_config_tissue_region_camelyon_subtype_cls_dataset-camelyon_model_type-gnn_architecture-tumor_regi_69ecdff7_mil_cached_training_method-function_20261026_020459.yaml"
    },
    "ResNet-50": {
        "eval": None,
        "history": None,
        "meta": None
    }
}
# Runs used with a caveat: marked † in every figure / table, the reason is printed as a footnote
# (multiclass runs as "multiclass <model>").
CAVEATS = {
    "ResNet-50": "selected checkpoint = epoch 1 of 11, where the model called every validation slide tumor (sensitivity 100 %, specificity 0 %); this already maximised the selection metric slide F2, and early stopping (patience 10) ended training while the validation AUC was still rising (61.7 -> 85.3 %). Its results reflect this checkpoint, not what the ResNet-50 encoder can reach."
}
# E4 center rotation: {corpus: {test center: {"eval": evaluation folder, "meta": training meta, "history": history}}}
# (folds trained with --split-phase test_c<t>; test center 4 = the seed_42 split)
RUNS_E4 = {
    "C-aug": {
        "0": {
            "eval": "results/evaluations/e4-cam-bin-aug-tc0_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_test_c0_20261030_225528",
            "history": "results/models/training_history_e4-cam-bin-aug-tc0_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_test_c0_segmentation.json",
            "meta": "results/metas/training_metas/e4-cam-bin-aug-tc0_task-lymph_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_dataset-camelyo_39f3977a_mil_cached_training_method-function_split_phase-test_c0_20261031_101732.yaml"
        },
        "1": {
            "eval": "results/evaluations/e4-cam-bin-aug-tc1_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_test_c1_20261030_152820",
            "history": "results/models/training_history_e4-cam-bin-aug-tc1_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_test_c1_segmentation.json",
            "meta": "results/metas/training_metas/e4-cam-bin-aug-tc1_task-lymph_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_dataset-camelyo_c4bfe60d_mil_cached_training_method-function_split_phase-test_c1_20261031_065848.yaml"
        },
        "2": {
            "eval": "results/evaluations/e4-cam-bin-aug-tc2_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_test_c2_20261029_222048",
            "history": "results/models/training_history_e4-cam-bin-aug-tc2_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_test_c2_segmentation.json",
            "meta": "results/metas/training_metas/e4-cam-bin-aug-tc2_task-lymph_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_dataset-camelyo_1fb908e3_mil_cached_training_method-function_split_phase-test_c2_20261030_200114.yaml"
        },
        "3": {
            "eval": "results/evaluations/e4-cam-bin-aug-tc3_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_test_c3_20261029_222004",
            "history": "results/models/training_history_e4-cam-bin-aug-tc3_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_test_c3_segmentation.json",
            "meta": "results/metas/training_metas/e4-cam-bin-aug-tc3_task-lymph_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_dataset-camelyo_adc11058_mil_cached_training_method-function_split_phase-test_c3_20261030_133702.yaml"
        },
        "4": {
            "eval": "results/evaluations/e2-cam-bin-gigapath_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_20261024_142805",
            "history": "results/models/training_history_e2-cam-bin-gigapath_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_segmentation.json",
            "meta": "results/metas/training_metas/e2-cam-bin-gigapath_task-lymph_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_dataset-camelyon_model_type-gnn_ar_78d582ac_mil_cached_training_method-function_20261025_093558.yaml"
        }
    },
    "C-noaug": {
        "0": {
            "eval": "results/evaluations/e4-cam-bin-noaug-tc0_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_test_c0_20261028_092839",
            "history": "results/models/training_history_e4-cam-bin-noaug-tc0_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_test_c0_segmentation.json",
            "meta": "results/metas/training_metas/e4-cam-bin-noaug-tc0_task-lymph_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_dataset-camel_71f2228d_mil_cached_training_method-function_split_phase-test_c0_20261028_232716.yaml"
        },
        "1": {
            "eval": "results/evaluations/e4-cam-bin-noaug-tc1_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_test_c1_20261028_092932",
            "history": "results/models/training_history_e4-cam-bin-noaug-tc1_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_test_c1_segmentation.json",
            "meta": "results/metas/training_metas/e4-cam-bin-noaug-tc1_task-lymph_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_dataset-camel_1f635105_mil_cached_training_method-function_split_phase-test_c1_20261028_164017.yaml"
        },
        "2": {
            "eval": "results/evaluations/e4-cam-bin-noaug-tc2_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_test_c2_20261028_171323",
            "history": "results/models/training_history_e4-cam-bin-noaug-tc2_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_test_c2_segmentation.json",
            "meta": "results/metas/training_metas/e4-cam-bin-noaug-tc2_task-lymph_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_dataset-camel_0a919ed1_mil_cached_training_method-function_split_phase-test_c2_20261029_045040.yaml"
        },
        "3": {
            "eval": "results/evaluations/e4-cam-bin-noaug-tc3_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_test_c3_20261029_074408",
            "history": "results/models/training_history_e4-cam-bin-noaug-tc3_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_test_c3_segmentation.json",
            "meta": "results/metas/training_metas/e4-cam-bin-noaug-tc3_task-lymph_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_dataset-camel_8fd8a7f0_mil_cached_training_method-function_split_phase-test_c3_20261029_203012.yaml"
        },
        "4": {
            "eval": "results/evaluations/e4-cam-bin-noaug-tc4_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_test_c4_20261029_074558",
            "history": "results/models/training_history_e4-cam-bin-noaug-tc4_lymph_dataset-camelyon_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_test_c4_segmentation.json",
            "meta": "results/metas/training_metas/e4-cam-bin-noaug-tc4_task-lymph_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_dataset-camel_2264d26d_mil_cached_training_method-function_split_phase-test_c4_20261029_192424.yaml"
        }
    }
}
# Folds with a caveat: {"<corpus> <test center>": reason}; marked † in the E4 figure and table.
E4_CAVEATS = {}
# Paired comparisons (name, model, reference): Δ = model − reference (skipped while a run is missing).
COMPARISONS = [("E2 encoder", "Prov-GigaPath", "ResNet-50"),
               ("E4 graph augmentation", "Prov-GigaPath", "Prov-GigaPath · C-noaug")]
DOMAIN_LONG = {"VAL": "Validation (center 3)", "TEST": "Test (center 4)"}
STAGES = ["itc", "micro", "macro"]                         # tumor sizes, ascending
STAGE_LONG = {"negative": "negative", "itc": "ITC", "micro": "micro", "macro": "macro"}
MC_ORDER = ["negative", "itc", "micro", "macro"]           # ordinal order for the multiclass metrics
N_VAL_GRAPHS = 79                                          # validation slides with a graph
N_BOOT = 2000
SEED = 0
SAVE_PREFIX = "cam"                                        # None = do not write figures / tables

## Data

In [ ]:
def _one(directory, pattern):
    hits = sorted(directory.glob(pattern)) if directory.is_dir() else []
    return hits[0] if hits else None


def resolve_path(path):
    # Repository-relative or absolute path; None stays None. A given path that does not exist is an error.
    if path is None:
        return None
    p = Path(path)
    p = p if p.is_absolute() else REPO_ROOT / p
    if not p.exists():
        raise FileNotFoundError(f"Path not found: {p}")
    return p


def find_key(d, key):
    if isinstance(d, dict):
        if key in d:
            return d[key]
        for v in d.values():
            hit = find_key(v, key)
            if hit is not None:
                return hit
    return None


cohorts = {}
for dom, split in (("VAL", "val"), ("TEST", "test")):
    c = pd.read_csv(SPLITS_DIR / f"{split}.csv")
    c["label"] = (c["tumor_status"] == "tumor").astype(int)
    c["stage"] = c["stage"].str.lower()
    cohorts[dom] = c[["slide_name", "patient_id", "center", "stage", "label"]].reset_index(drop=True)


def val_counts(slide, n=N_VAL_GRAPHS):
    # Reconstruct the validation confusion counts from the stored rates (exact: they must come out as integers).
    n_pos = slide["validation_label_positive_rate"] * n
    tp, tn = slide["sensitivity"] * n_pos, slide["specificity"] * (n - n_pos)
    counts = {"tp": tp, "fn": n_pos - tp, "tn": tn, "fp": n - n_pos - tn}
    if any(abs(v - round(v)) > 1e-6 for v in counts.values()) or \
            abs((tp + counts["fp"]) - slide["validation_pred_positive_rate"] * n) > 1e-6:
        raise ValueError(f"Validation counts are not consistent with N_VAL_GRAPHS = {n}: {counts}")
    return {k: int(round(v)) for k, v in counts.items()}


results, val_info, availability = {}, {}, []
for model, paths in RUNS.items():
    eval_dir = resolve_path(paths.get("eval"))
    if eval_dir is None:
        availability.append({"Model": model, "Status": "not available yet", "Test slides": "–", "Source": "–"})
        continue
    vt = json.load(open(eval_dir / "validation_thresholds" / "validation_thresholds.json"))
    val_info[model] = {"threshold": vt["slide"]["threshold"], "counts": val_counts(vt["slide"]),
                       "auc": vt["validation_slide_auroc"]}
    split_dir = eval_dir / "split_test"
    preds = pd.read_csv(_one(split_dir, "*_test_predictions.csv"))[["slide_name", "prob", "label"]]
    if preds["slide_name"].duplicated().any():
        raise ValueError(f"{split_dir} predictions are not slide-level")
    metrics = json.load(open(_one(split_dir, "*_test_slide_metrics.json")))
    merged = cohorts["TEST"].merge(preds, on="slide_name", how="left", suffixes=("", "_pred"))
    known = merged["prob"].notna()
    if (merged.loc[known, "label"] != merged.loc[known, "label_pred"]).any():
        raise ValueError(f"Label mismatch between split file and predictions: {model}")
    results[model] = {"prob": merged["prob"].to_numpy(), "at_val": metrics["at_validation_threshold"]}
    availability.append({"Model": model, "Status": "used, with caveat †" if model in CAVEATS else "used",
                         "Test slides": f"{known.sum()}/{len(merged)}", "Source": eval_dir.name[:60] + "…"})

colors = {m: IEEE_COLORS[i % len(IEEE_COLORS)] for i, m in enumerate(RUNS)}
# Redundant encoding for colour-blind readers: every model also has its own marker and line style.
markers = {m: ["o", "s", "^", "D", "v", "P"][i % 6] for i, m in enumerate(RUNS)}
lines = {m: ["-", "--", "-.", ":"][i % 4] for i, m in enumerate(RUNS)}
models = [m for m in RUNS if m in results]


def lab(model, task="binary"):
    # Display name; † marks a run with a caveat (footnote in every figure that shows it).
    key = model if task == "binary" else f"multiclass {model}"
    return f"{model}†" if key in CAVEATS else model
y_test = cohorts["TEST"]["label"].to_numpy()
pd.DataFrame(availability).set_index("Model")

In [ ]:
# Multiclass predictions. The evaluation stores class ids in alphabetical order (itc, macro, micro, negative);
# they are remapped to the ordinal order negative < ITC < micro < macro used for all multiclass metrics.
mc_results, mc_val, mc_availability = {}, {}, []
for model, paths in RUNS_MC.items():
    eval_dir = resolve_path(paths.get("eval"))
    if eval_dir is None:
        mc_availability.append({"Model": model, "Status": "not available yet"})
        continue
    vm = json.load(open(eval_dir / "validation_thresholds" / "validation_thresholds.json"))["validation_multiclass"]
    mc_val[model] = {"Macro-F1": vm["f1_macro"], "AUROC (OvO)": vm["auc_ovo_macro"]}
    split_dir = eval_dir / "split_test"
    stored = json.load(open(_one(split_dir, "*_test_multiclass_metrics.json")))
    names = stored["class_names"]
    if isinstance(names, str):
        names = [n.strip(" '\"") for n in names.strip("[]").split(",")]
    preds = pd.read_csv(_one(split_dir, "*_test_predictions.csv"))
    merged = cohorts["TEST"].merge(preds, on="slide_name", how="left", suffixes=("", "_pred"))
    known = merged["prob"].notna()
    if (merged.loc[known, "label_pred"].astype(int).map(dict(enumerate(names))) != merged.loc[known, "stage"]).any():
        raise ValueError(f"Class mismatch between split file and predictions: {model}")
    P = np.column_stack([merged[f"prob_class_{names.index(name)}"].to_numpy() for name in MC_ORDER])
    mc_results[model] = {"P": P, "y": merged["stage"].map(MC_ORDER.index).to_numpy(), "stored": stored}
    mc_availability.append({"Model": model, "Status": "used, with caveat †" if f"multiclass {model}" in CAVEATS else "used"})
mc_models = [m for m in RUNS_MC if m in mc_results]
mc_colors = {m: colors.get(m, IEEE_COLORS[i % len(IEEE_COLORS)]) for i, m in enumerate(RUNS_MC)}
mc_markers = {m: markers.get(m, "o") for m in RUNS_MC}
pd.DataFrame(mc_availability).set_index("Model")

In [ ]:
# Training metas: graph corpus and patch-node limit per run; excluded runs with their reason
for task, runs in (("binary", RUNS), ("multiclass", RUNS_MC)):
    for model, paths in runs.items():
        meta = resolve_path(paths.get("meta"))
        m = yaml.safe_load(open(meta)) if meta else {}
        corpus = find_key(m, "graph_target_dir")
        limit = find_key(m, "max_patch_nodes_for_full_forward")
        print(f"{task:10s} {model:24s} corpus: {Path(str(corpus)).name if corpus else 'not trained yet':58s} "
              f"max_patch_nodes: {limit if limit is not None else '–'}")
for model, reason in CAVEATS.items():
    print(f"\n† {model}: {reason}")

In [ ]:
# Cohort description (split files; the slides without a graph are excluded from all metrics)
rows = []
for dom, c in cohorts.items():
    row = {"Center": DOMAIN_LONG[dom], "Slides": len(c), "Patients": c["patient_id"].nunique(),
           "Tumor": int(c["label"].sum()), "Normal": int((1 - c["label"]).sum())}
    row.update({STAGE_LONG[s]: int((c["stage"] == s).sum()) for s in STAGES})
    rows.append(row)
cohort_table = pd.DataFrame(rows).set_index("Center")
cohort_table

In [ ]:
# Protocol checks: the test confusion matrix recomputed at the validation threshold equals the stored one, and
# the test evaluation uses the validation threshold of the same checkpoint.
def confusion(y, p, t):
    pred = (p > t).astype(int)
    return {"tp": int(((pred == 1) & (y == 1)).sum()), "fn": int(((pred == 0) & (y == 1)).sum()),
            "fp": int(((pred == 1) & (y == 0)).sum()), "tn": int(((pred == 0) & (y == 0)).sum())}


for model, r in results.items():
    ok = ~np.isnan(r["prob"])
    cm = confusion(y_test[ok], r["prob"][ok], r["at_val"]["threshold"])
    same_thr = abs(r["at_val"]["threshold"] - val_info[model]["threshold"]) < 1e-9
    ok_cm = cm == {k: r["at_val"][k] for k in cm}
    print(f"{model:24s} threshold={r['at_val']['threshold']:.4f}  VAL {val_info[model]['counts']}  TEST {cm}  "
          f"{'ok' if same_thr and ok_cm else 'MISMATCH'}")

## 1 · Training curves

Validation curves (center 3). The vertical line marks the saved checkpoint (binary: max. validation slide F2;
multiclass: max. validation quadratic κ; ties → last epoch, as in `train.py`), i.e. the weights evaluated
below. Runs with a caveat are marked † (footnote).

In [ ]:
apply_ieee_style()
W, _ = figsize("double")


def save(fig, name, shown=None):
    # shown: display names in the figure; a † run adds its reason as a footnote below the figure.
    notes = [f"† {m}: {r}" for m, r in CAVEATS.items()
             if shown is None or any(lab(m) in s or lab(m.replace("multiclass ", ""), "multiclass") in s for s in shown)]
    if notes:
        import textwrap
        fig.text(0.01, -0.01, "\n".join(textwrap.fill(n, 190) for n in notes), fontsize=6, color="0.3",
                 ha="left", va="top")
    if SAVE_PREFIX:
        for ext in ("png", "pdf"):
            fig.savefig(OUT_DIR / f"{SAVE_PREFIX}_{name}.{ext}", bbox_inches="tight")
        print("Saved:", OUT_DIR / f"{SAVE_PREFIX}_{name}.png")


def panel(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left")


def load_histories(runs):
    out = {}
    for model, paths in runs.items():
        f = resolve_path(paths.get("history"))
        if f is not None:
            out[model] = json.load(open(f))
    return out


def selected_epoch(hist):
    meta = hist.get("metadata", {})
    metric = meta.get("monitor_metric")
    if metric:
        values = np.asarray(hist.get(f"val_{metric}") or hist[metric], dtype=float)
        best = values.max() if meta.get("monitor_mode", "max") == "max" else values.min()
        return int(np.flatnonzero(values == best)[-1])
    return int(np.argmin(hist["val_loss"]))


histories, mc_histories = load_histories(RUNS), load_histories(RUNS_MC)
ROWS = [("binary", histories, lambda m: lab(m),
         [("val_slide_auc", "Slide AUC (%)"), ("val_slide_sensitivity", "Slide sensitivity (%)"),
          ("val_slide_f2", "Slide F2 (%), selection"), ("loss", "Loss")]),
        ("multiclass", mc_histories, lambda m: lab(m, "multiclass"),
         [("val_auc", "AUROC (%)"), ("val_f1", "Macro-F1 (%)"),
          ("val_kappa_quadratic", "Quadratic κ (%), selection"), ("loss", "Loss")])]
fig, axes = plt.subplots(2, 4, figsize=(W, 4.6))
letters = iter("abcdefgh")
legend_entries = {}
for row_axes, (task, hists, label_of, curves) in zip(axes, ROWS):
    for ax, (key, title) in zip(row_axes, curves):
        for model, hist in hists.items():
            c = colors.get(model, mc_colors.get(model))
            mk, sel = markers.get(model, "o"), selected_epoch(hist)
            legend_entries.setdefault(label_of(model), (c, mk))
            if key == "loss":
                for split, ls, alpha in (("train", "--", 0.55), ("val", "-", 1.0)):
                    v = hist[f"{split}_loss"]
                    ax.plot(np.arange(1, len(v) + 1), v, color=c, linestyle=ls, alpha=alpha, linewidth=1.2,
                            marker=mk, markevery=4, markersize=3, markerfacecolor="white" if split == "train" else c)
            elif hist.get(key):
                v = np.asarray(hist[key], dtype=float)
                v = 100 * v if v.max() <= 1.0 else v
                ax.plot(np.arange(1, len(v) + 1), v, color=c, linewidth=1.3, marker=mk, markevery=4, markersize=3)
                ax.plot(sel + 1, v[sel], mk, color=c, markersize=6, markeredgecolor="black", markeredgewidth=0.7,
                        zorder=4)
            ax.axvline(sel + 1, color=c, linestyle="-.", linewidth=0.8, alpha=0.8)
        ax.set_title(f"({next(letters)}) {title}", loc="left", fontsize=8)
        ax.set_xlabel("Epoch")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=5))
        if not hists:
            ax.text(0.5, 0.5, f"no {task} run yet", transform=ax.transAxes, ha="center", va="center",
                    fontsize=7, color="0.5")
    row_axes[0].set_ylabel(task, fontsize=8, fontweight="bold")
handles = [Line2D([], [], color=c, marker=mk, markersize=4, linewidth=1.5, label=label)
           for label, (c, mk) in legend_entries.items()]
handles += [Line2D([], [], color="0.3", linestyle="--", alpha=0.55, marker="o", markerfacecolor="white",
                   markersize=3, label="training loss (hollow markers)"),
            Line2D([], [], color="0.3", linestyle="-.", marker="o", markerfacecolor="white", markeredgecolor="black",
                   markersize=6, label="selected checkpoint (line, black-edged marker)")]
fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.0))
fig.tight_layout(rect=(0, 0.1, 1, 1))
save(fig, "training", shown=list(legend_entries))
plt.show()
for task, hists in (("binary", histories), ("multiclass", mc_histories)):
    for model, hist in hists.items():
        print(f"{task:10s} {model:24s} selected epoch {selected_epoch(hist) + 1} / {len(hist['train_loss'])}")

## Bootstrap estimates (test center)

In [ ]:
rng = np.random.default_rng(SEED)
patients, inverse = np.unique(cohorts["TEST"]["patient_id"].to_numpy(), return_inverse=True)
rows_of = [np.flatnonzero(inverse == k) for k in range(len(patients))]
boot = [np.concatenate([rows_of[k] for k in rng.integers(0, len(patients), len(patients))]) for _ in range(N_BOOT)]


def rates_from_counts(cm):
    sens = cm["tp"] / (cm["tp"] + cm["fn"]) if cm["tp"] + cm["fn"] else np.nan
    spec = cm["tn"] / (cm["tn"] + cm["fp"]) if cm["tn"] + cm["fp"] else np.nan
    rates = [v for v in (sens, spec) if not np.isnan(v)]
    return {"Sensitivity": sens, "Specificity": spec, "Balanced accuracy": np.mean(rates) if rates else np.nan,
            "PPV": cm["tp"] / (cm["tp"] + cm["fp"]) if cm["tp"] + cm["fp"] else np.nan,
            "NPV": cm["tn"] / (cm["tn"] + cm["fn"]) if cm["tn"] + cm["fn"] else np.nan,
            "F1": 2 * cm["tp"] / (2 * cm["tp"] + cm["fp"] + cm["fn"]) if cm["tp"] + cm["fp"] + cm["fn"] else np.nan}


def binary_metrics(y, p, t):
    ok = ~np.isnan(p)
    y, p = y[ok], p[ok]
    return {"AUC": roc_auc_score(y, p) if 0 < y.sum() < len(y) else np.nan, **rates_from_counts(confusion(y, p, t))}


def ci(x):
    x = pd.Series(x).dropna()
    return np.percentile(x, [2.5, 97.5]) if len(x) else np.array([np.nan, np.nan])


test_est, val_est = {}, {}
for model in results:
    r, t = results[model], results[model]["at_val"]["threshold"]
    test_est[model] = (binary_metrics(y_test, r["prob"], t),
                       pd.DataFrame([binary_metrics(y_test[i], r["prob"][i], t) for i in boot]))
    val_est[model] = {"AUC": val_info[model]["auc"], **rates_from_counts(val_info[model]["counts"])}


def gap(model):
    # Gap = AUC(Val) − AUC(Test); CI from the test-center bootstrap (validation value fixed).
    v = val_est[model]["AUC"]
    return v - test_est[model][0]["AUC"], ci(v - test_est[model][1]["AUC"].to_numpy())


gaps = {m: gap(m) for m in models}
print(f"{N_BOOT} patient-level resamples of the test center ({len(patients)} patients).")

## 2 · AUC on both centers, Gap, ROC

(a) Slide-level AUC on the validation center (hollow, point estimate, biased by checkpoint selection) and on
the independent test center (filled, 95 % CI). (b) Gap = AUC(Val) − AUC(Test) with 95 % CI (test-center
sampling only); positive = worse on the test center. (c) ROC curves on the test center; dot = operating point at
the validation threshold.

In [ ]:
if not models:
    print("No usable binary run.")
else:
    fig, (ax_a, ax_b, ax_c) = plt.subplots(1, 3, figsize=(W, 2.7))
    dodge = dict(zip(models, np.linspace(-0.12, 0.12, len(models)) if len(models) > 1 else [0.0]))
    for model in models:
        c, x0, x1 = colors[model], dodge[model], 1 + dodge[model]
        v, (pt, reps) = val_est[model]["AUC"], test_est[model]
        lo, hi = 100 * ci(reps["AUC"])
        ax_a.plot([x0, x1], [100 * v, 100 * pt["AUC"]], color=c, linestyle=lines[model], linewidth=1.2, alpha=0.85)
        ax_a.plot([x1, x1], [lo, hi], color=c, linewidth=1.2, alpha=0.75)
        ax_a.plot(x0, 100 * v, markers[model], markerfacecolor="white", markeredgecolor=c, markeredgewidth=1.3,
                  markersize=6, zorder=4)
        ax_a.plot(x1, 100 * pt["AUC"], markers[model], color=c, markersize=6, zorder=4)
    ax_a.set_xticks([0, 1], ["Validation\n(center 3)", "Test\n(center 4)"])
    ax_a.set_xlim(-0.45, 1.45)
    ax_a.set_ylabel("Slide-level AUC (%)")
    ax_a.grid(axis="x", visible=False)
    panel(ax_a, "a", "AUC")

    order = sorted(models, key=lambda m: gaps[m][0])
    for yi, model in enumerate(order):
        g, (lo, hi) = gaps[model]
        ax_b.plot([100 * lo, 100 * hi], [yi, yi], color=colors[model], linewidth=2.0, solid_capstyle="round")
        ax_b.plot(100 * g, yi, markers[model], color=colors[model], markersize=6.5, markeredgecolor="white", zorder=4)
        ax_b.annotate(f"{100 * g:+.1f}", (100 * g, yi), xytext=(0, 6), textcoords="offset points", ha="center",
                      va="bottom", fontsize=6.5, color="0.2")
    ax_b.axvline(0, color="0.3", linewidth=0.9, linestyle=":")
    ax_b.set_yticks(range(len(order)), [lab(m) for m in order])
    ax_b.set_ylim(-0.6, len(order) - 0.4)
    ax_b.set_xlabel("Gap, Val − Test (pp, 95 % CI)")
    ax_b.grid(axis="y", visible=False)
    panel(ax_b, "b", "Gap")

    ax_c.plot([0, 1], [0, 1], color="0.6", linestyle=":", linewidth=0.8)
    for model in models:
        p, pt = results[model]["prob"], test_est[model][0]
        ok = ~np.isnan(p)
        fpr, tpr, _ = roc_curve(y_test[ok], p[ok])
        ax_c.plot(fpr, tpr, color=colors[model], linewidth=1.4, drawstyle="steps-post", linestyle=lines[model])
        ax_c.plot(1 - pt["Specificity"], pt["Sensitivity"], markers[model], color=colors[model], markersize=5.5,
                  markeredgecolor="white", markeredgewidth=0.8, zorder=4)
    ax_c.set_xlabel("1 − specificity")
    ax_c.set_ylabel("Sensitivity")
    ax_c.set_aspect("equal")
    panel(ax_c, "c", "ROC, test center")
    fig.legend(handles=[Line2D([], [], color=colors[m], marker=markers[m], linestyle=lines[m], markersize=5,
                               label=lab(m)) for m in models]
               + [Line2D([], [], marker="o", markerfacecolor="white", markeredgecolor="0.3", linestyle="",
                         label="(a) hollow: validation (no CI)"),
                  Line2D([], [], marker="o", color="0.3", linestyle="", label="(a) filled: test"),
                  Line2D([], [], color="0.4", marker="|", markersize=10, markeredgewidth=1.2, linestyle="",
                         label="(a, b) bar: 95 % CI (test-center sampling)"),
                  Line2D([], [], color="0.3", linestyle=":", label="(b) no gap;  (c) chance level"),
                  Line2D([], [], color="0.3", marker="o", linestyle="", markersize=4,
                         label="(c) dot: operating point at the validation threshold")],
               loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.0))
    fig.tight_layout(rect=(0, 0.2, 1, 1))
    save(fig, "auc_gap_roc", shown=[lab(m) for m in models])
    plt.show()

## 3 · Sensitivity and specificity at the validation threshold

Tumor slides split into detected (TP) and missed (FN), normal slides into cleared (TN) and false alarms (FP),
per center. The threshold was chosen on the validation center (G-mean), so the validation rates are in-sample;
the test rates show how the threshold transfers to a new center. Numbers are slide counts.

In [ ]:
def counts_of(model, dom):
    if dom == "VAL":
        return val_info[model]["counts"]
    ok = ~np.isnan(results[model]["prob"])
    return confusion(y_test[ok], results[model]["prob"][ok], results[model]["at_val"]["threshold"])


if models:
    order = models[::-1]
    BAR, GAP_Y = 0.34, 0.2
    fig, axes = plt.subplots(1, 2, figsize=(W, 0.62 * len(models) + 1.15), sharey=True)
    yticks, yticklabels = [], []
    for ax, (good, bad, cls, title, letter) in zip(axes, [("tp", "fn", 1, "Tumor slides", "a"),
                                                          ("tn", "fp", 0, "Normal slides", "b")]):
        for yi, model in enumerate(order):
            c = colors[model]
            for dom, dy in (("VAL", GAP_Y), ("TEST", -GAP_Y)):
                y = yi + dy
                if ax is axes[0]:
                    yticks.append(y)
                    yticklabels.append("Val" if dom == "VAL" else "Test")
                cm = counts_of(model, dom)
                share = 100 * cm[good] / (cm[good] + cm[bad])
                ax.barh(y, share, height=BAR, color=c, edgecolor="white", linewidth=0.6)
                ax.barh(y, 100 - share, left=share, height=BAR, color=c, alpha=0.18, hatch="////", edgecolor=c,
                        linewidth=0)
                if share > 18:
                    ax.text(share - 1.5, y, f"{good.upper()} {cm[good]}", ha="right", va="center", fontsize=6.5,
                            color="white")
                ax.text(101.5, y, f"{share:.0f} %", ha="left", va="center", fontsize=6.5, color="0.1",
                        fontweight="bold")
                ax.text(120, y, f"{bad.upper()} {cm[bad]}", ha="left", va="center", fontsize=6.5,
                        color="0.1" if cm[bad] else "0.55")
        ax.set_xlim(0, 134)
        ax.text(101.5, 1.0, "rate", transform=ax.get_xaxis_transform(), ha="left", va="bottom", fontsize=6.5,
                color="0.4")
        ax.text(120, 1.0, "errors", transform=ax.get_xaxis_transform(), ha="left", va="bottom", fontsize=6.5,
                color="0.4")
        ax.set_xticks([0, 25, 50, 75, 100])
        ax.set_xlabel("% of slides in class")
        ax.grid(axis="y", visible=False)
        ax.set_title(f"({letter}) {title} ({'sensitivity' if cls else 'specificity'})", loc="left", pad=14)
    axes[0].set_yticks(yticks, yticklabels, fontsize=7)
    axes[0].set_ylim(-0.6, len(order) - 0.4)
    fig.canvas.draw()
    label_w = max(t.get_window_extent(fig.canvas.get_renderer()).width
                  for t in axes[0].get_yticklabels()) * 72 / fig.dpi
    for yi, model in enumerate(order):
        axes[0].annotate(lab(model), xy=(0, yi), xycoords=("axes fraction", "data"), xytext=(-(label_w + 8), 0),
                         textcoords="offset points", ha="right", va="center", fontsize=8, color=colors[model],
                         fontweight="bold")
        for ax in axes:
            if yi:
                ax.axhline(yi - 0.5, color="0.85", linewidth=0.6)
    fig.legend(handles=[Patch(facecolor="0.35", label="correctly classified (TP / TN)"),
                        Patch(facecolor="0.35", alpha=0.18, hatch="////", edgecolor="0.35", linewidth=0,
                              label="misclassified (FN / FP), count under \"errors\"")],
               loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.0))
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    save(fig, "operating_point", shown=[lab(m) for m in models])
    plt.show()

## 4 · Threshold transfer (test center)

(a) Slide scores on the test center per class with the validation threshold (dashed): it shows whether errors
come from overlapping scores (ranking) or from a threshold that sits at the wrong place for the new center.
(b) Balanced accuracy at the validation threshold (marker) vs. the **oracle threshold** chosen on the test
center itself (bar end, diagnostic only, optimistic): the difference is the cost of transferring the
threshold, the remaining distance to 100 % is the ranking error.

In [ ]:
def oracle_balanced_accuracy(y, p):
    ok = ~np.isnan(p)
    y, p = y[ok], p[ok]
    best = (0.0, np.nan)
    for t in np.concatenate([[-np.inf], np.unique(p)]):
        ba = binary_metrics(y, p, t)["Balanced accuracy"]
        if ba > best[0]:
            best = (ba, t)
    return best


oracle = {m: oracle_balanced_accuracy(y_test, results[m]["prob"]) for m in models}
if models:
    fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(W, 2.6), gridspec_kw={"width_ratios": [1.3, 1]})
    jit = np.random.default_rng(SEED)
    for k, model in enumerate(models):
        c, p = colors[model], results[model]["prob"]
        for cls, dx, face in ((0, -0.18, "white"), (1, 0.18, c)):
            m = (y_test == cls) & ~np.isnan(p)
            ax_a.scatter(k + dx + jit.uniform(-0.1, 0.1, m.sum()), p[m], s=10, facecolor=face, edgecolor=c,
                         linewidth=0.8, zorder=3, marker=markers[model])
            ax_a.plot([k + dx - 0.14, k + dx + 0.14], [np.median(p[m])] * 2, color="0.15", linewidth=1.4, zorder=4)
        t = results[model]["at_val"]["threshold"]
        ax_a.plot([k - 0.42, k + 0.42], [t, t], color="0.25", linestyle="--", linewidth=0.9)
    ax_a.set_xticks(range(len(models)), [lab(m) for m in models])
    ax_a.set_xlim(-0.6, len(models) - 0.4)
    ax_a.set_ylim(-0.03, 1.03)
    ax_a.set_ylabel("Slide score")
    ax_a.grid(axis="x", visible=False)
    panel(ax_a, "a", "Test-center scores")

    order = models[::-1]
    for yi, model in enumerate(order):
        c = colors[model]
        ba_val, ba_orc = test_est[model][0]["Balanced accuracy"], oracle[model][0]
        ax_b.plot([100 * ba_val, 100 * ba_orc], [yi, yi], color=c, linewidth=2.4, alpha=0.35, solid_capstyle="butt")
        ax_b.plot(100 * ba_orc, yi, marker="|", color=c, markersize=10, markeredgewidth=1.6, zorder=4)
        ax_b.plot(100 * ba_val, yi, markers[model], color=c, markersize=6, zorder=5)
        ax_b.annotate(f"−{100 * (ba_orc - ba_val):.1f}", (100 * ba_val, yi), xytext=(-6, 0), textcoords="offset points",
                      ha="right", va="center", fontsize=6.5, color="0.3")
    ax_b.set_yticks(range(len(order)), [lab(m) for m in order])
    ax_b.set_ylim(-0.6, len(order) - 0.4)
    ax_b.set_xlim(40, 102)
    ax_b.set_xlabel("Balanced accuracy, test center (%)")
    ax_b.grid(axis="y", visible=False)
    panel(ax_b, "b", "Cost of the transferred threshold")
    fig.legend(handles=[
        Line2D([], [], marker="o", markerfacecolor="white", markeredgecolor="0.3", linestyle="", label="(a) normal"),
        Line2D([], [], marker="o", color="0.3", linestyle="", label="(a) tumor"),
        Line2D([], [], color="0.15", linewidth=1.4, label="(a) median"),
        Line2D([], [], color="0.25", linestyle="--", label="(a) validation threshold"),
        Line2D([], [], marker="o", color="0.3", linestyle="", label="(b) at validation threshold"),
        Line2D([], [], marker="|", color="0.3", linestyle="", markersize=9, markeredgewidth=1.6,
               label="(b) oracle threshold (diagnostic)")],
        loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.0))
    fig.tight_layout(rect=(0, 0.17, 1, 1))
    save(fig, "threshold_transfer", shown=[lab(m) for m in models])
    plt.show()

## 5 · Detection by metastasis size (test center, table)

Sensitivity at the validation threshold per metastasis size and specificity on the normal slides, with 95 %
CI and the number of slides (with a graph). ITC ≤ 0.2 mm, micro ≤ 2 mm, macro > 2 mm. The small counts make the
intervals wide; differences between sizes or models below roughly 30 pp are not resolvable here.

In [ ]:
rows = []
for model in models:
    r, t = results[model], results[model]["at_val"]["threshold"]
    row = {"Model": lab(model)}
    for stage, metric in [("negative", "Specificity")] + [(s, "Sensitivity") for s in STAGES]:
        mask = (cohorts["TEST"]["stage"] == stage).to_numpy() & ~np.isnan(r["prob"])
        point = binary_metrics(y_test[mask], r["prob"][mask], t)[metric]
        reps = [binary_metrics(y_test[i[mask[i]]], r["prob"][i[mask[i]]], t)[metric] for i in boot]
        lo, hi = 100 * ci(reps)
        row[f"{STAGE_LONG[stage]} ({'spec.' if stage == 'negative' else 'sens.'})"] = \
            f"{100 * point:.0f} [{lo:.0f}, {hi:.0f}] (n = {int(mask.sum())})"
    rows.append(row)
size_table = pd.DataFrame(rows).set_index("Model") if rows else pd.DataFrame()
size_table.columns.name = "% [95 % CI] (slides)"
size_table

## 6 · Tumor localization on the test center (region and patch level)

The graph model also scores every region and every patch. Labels come from the CAMELYON17 lesion annotations
of the test center (positive = inside an annotated metastasis; negative = outside on annotated slides and all
patches / regions of normal slides; unlabeled ones are not scored). Only about 1 % of the regions and patches are
positive, so the **precision–recall curve and average precision (AP)** are the informative measures (dashed
line = prevalence = AP of a random scorer); the ROC AUC is dominated by the many easy negatives and is shown
alongside. Region level: 95 % CI from the same patient-level bootstrap; patch level (2.4 M patches): point
estimates. Dice at the fixed 0.5 cut-off is listed in the table (threshold-dependent).

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve

LEVELS = {"region": ("region_prob", "region_label"), "patch": ("patch_prob", "patch_label")}
loc = {}
for model in models:
    split_dir = resolve_path(RUNS[model]["eval"]) / "split_test"
    for level, (pcol, ycol) in LEVELS.items():
        f = _one(split_dir, f"*_test_{level}_predictions.csv")
        if f is None:
            continue
        d = pd.read_csv(f, usecols=["slide_name", pcol, ycol])
        d = d[d[ycol].isin([0, 1])]
        stored = json.load(open(_one(split_dir, f"*_test_{level}_metrics.json")))
        loc[(model, level)] = {"y": d[ycol].to_numpy().astype(int), "p": d[pcol].to_numpy(),
                               "slide": d["slide_name"].to_numpy(), "stored": stored}


def loc_scores(y, p):
    ok = ~np.isnan(p)
    y, p = y[ok], p[ok]
    if not 0 < y.sum() < len(y):
        return {"AUC": np.nan, "AP": np.nan}
    return {"AUC": roc_auc_score(y, p), "AP": average_precision_score(y, p)}


# patient-level bootstrap for the region level: resample patients, take all their regions
slide_patient = dict(zip(cohorts["TEST"]["slide_name"], cohorts["TEST"]["patient_id"]))
loc_est = {}
for (model, level), d in loc.items():
    point = loc_scores(d["y"], d["p"])
    reps = None
    if level == "region":
        pat = np.array([slide_patient[s] for s in d["slide"]])
        rows_by_patient = {pt: np.flatnonzero(pat == pt) for pt in patients}
        reps = pd.DataFrame([loc_scores(d["y"][idx], d["p"][idx]) for idx in
                             (np.concatenate([rows_by_patient[patients[k]] for k in
                                              np.random.default_rng(SEED + b).integers(0, len(patients), len(patients))])
                              for b in range(N_BOOT))])
    loc_est[(model, level)] = (point, reps)

loc_models = [m for m in models if (m, "region") in loc]
if not loc_models:
    print("No region / patch predictions available.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(W, 2.7), gridspec_kw={"width_ratios": [1, 1, 1.15]})
    for ax, level, letter in zip(axes[:2], LEVELS, "ab"):
        prev = None
        for model in loc_models:
            if (model, level) not in loc:
                continue
            d = loc[(model, level)]
            prec, rec, _ = precision_recall_curve(d["y"], d["p"])
            ax.plot(rec, prec, color=colors[model], linestyle=lines[model], linewidth=1.4, drawstyle="steps-post")
            prev = d["y"].mean()
        if prev is not None:
            ax.axhline(prev, color="0.5", linestyle="--", linewidth=0.8)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.02)
        ax.set_xlabel("Recall (sensitivity)")
        ax.set_aspect("equal")
        n_pos = int(loc[(loc_models[0], level)]["y"].sum())
        n_all = len(loc[(loc_models[0], level)]["y"])
        panel(ax, letter, f"{level.capitalize()} level: PR")
        ax.text(0.05, 0.32, f"{n_pos:,} of {n_all:,}\n{dict(region='regions', patch='patches')[level]} positive".replace(",", " "), transform=ax.transAxes,
                fontsize=6.5, color="0.3", ha="left", va="center",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1))
    axes[0].set_ylabel("Precision")

    ax = axes[2]
    combos = [(level, metric) for level in LEVELS for metric in ("AP", "AUC")]
    offs = dict(zip(loc_models, np.linspace(-0.15, 0.15, len(loc_models)) if len(loc_models) > 1 else [0.0]))
    for model in loc_models:
        for j, (level, metric) in enumerate(combos):
            if (model, level) not in loc_est:
                continue
            point, reps = loc_est[(model, level)]
            x = j + offs[model]
            if reps is not None:
                lo, hi = 100 * ci(reps[metric])
                ax.plot([x, x], [lo, hi], color=colors[model], linewidth=1.2, alpha=0.75)
            ax.plot(x, 100 * point[metric], markers[model], color=colors[model], markersize=6, zorder=4)
    ax.set_xticks(range(len(combos)), [f"{lvl}\n{m}" for lvl, m in combos])
    ax.set_ylim(-3, 103)
    ax.set_ylabel("%")
    ax.grid(axis="x", visible=False)
    ax.axvline(1.5, color="0.75", linewidth=0.8)
    panel(ax, "c", "AP and ROC AUC")
    fig.legend(handles=[Line2D([], [], color=colors[m], marker=markers[m], linestyle=lines[m], markersize=5,
                               label=lab(m)) for m in loc_models]
               + [Line2D([], [], color="0.5", linestyle="--", label="(a, b) prevalence = AP of a random scorer"),
                  Line2D([], [], color="0.4", marker="|", markersize=10, markeredgewidth=1.2, linestyle="",
                         label="(c) bar: 95 % CI (region level; patch level: point estimate)")],
               loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.0))
    fig.tight_layout(rect=(0, 0.17, 1, 1))
    save(fig, "localization", shown=[lab(m) for m in loc_models])
    plt.show()

rows = []
for model in loc_models:
    for level in LEVELS:
        if (model, level) not in loc_est:
            continue
        point, reps = loc_est[(model, level)]
        st = loc[(model, level)]["stored"]["threshold_0_5"]
        cell = (lambda k: f"{100 * point[k]:.1f}" + (f" [{100 * ci(reps[k])[0]:.1f}, {100 * ci(reps[k])[1]:.1f}]"
                                                     if reps is not None else ""))
        rows.append({"Model": lab(model), "Level": level, "Positives": int(loc[(model, level)]["y"].sum()),
                     "Scored": len(loc[(model, level)]["y"]), "AP": cell("AP"), "ROC AUC": cell("AUC"),
                     "Dice @ 0.5": f"{100 * st['dice']:.1f}", "Sens. @ 0.5": f"{100 * st['sensitivity']:.1f}",
                     "Spec. @ 0.5": f"{100 * st['specificity']:.1f}"})
loc_table = pd.DataFrame(rows).set_index(["Model", "Level"]) if rows else pd.DataFrame()
loc_table.columns.name = "% [95 % CI]"
loc_table

## 7 · Multiclass: negative / ITC / micro / macro

Prevalence-robust metrics only: one-vs-one macro AUROC (Hand & Till, threshold-free), macro-F1 and the
**quadratic-weighted κ** on the ordinal scale negative < ITC < micro < macro (the CAMELYON17 metric). κ is
computed here for the test center, because the stored value in `*_multiclass_metrics.json` uses alphabetical
class ids and is not ordinal; the validation κ is the (ordinal) training value at the selected epoch.
Validation = point estimate, test = 95 % CI. Right: test-center confusion matrix (slide counts).

In [ ]:
MC_METRICS = ["AUROC (OvO)", "Macro-F1", "Quadratic κ"]


def mc_metrics(y, P):
    ok = ~np.isnan(P).any(axis=1)
    y, P = y[ok], P[ok]
    pred, labels = P.argmax(axis=1), list(range(len(MC_ORDER)))
    out = {"Macro-F1": f1_score(y, pred, labels=labels, average="macro", zero_division=0),
           "Quadratic κ": cohen_kappa_score(y, pred, labels=labels, weights="quadratic"), "AUROC (OvO)": np.nan}
    if all((y == k).any() for k in labels):
        out["AUROC (OvO)"] = roc_auc_score(y, P / P.sum(axis=1, keepdims=True), multi_class="ovo",
                                           average="macro", labels=labels)
    return out


mc_test, mc_valp = {}, {}
for model in mc_models:
    r = mc_results[model]
    mc_test[model] = (mc_metrics(r["y"], r["P"]), pd.DataFrame([mc_metrics(r["y"][i], r["P"][i]) for i in boot]))
    point = dict(mc_val[model])
    if model in mc_histories:
        point["Quadratic κ"] = mc_histories[model]["val_kappa_quadratic"][selected_epoch(mc_histories[model])] / 100
    mc_valp[model] = point

if not mc_models:
    print("No usable multiclass run.")
else:
    fig = plt.figure(figsize=(W, 2.9))
    gs = fig.add_gridspec(1, 1 + len(mc_models), width_ratios=[1.4] + [1] * len(mc_models))
    ax = fig.add_subplot(gs[0])
    offs = dict(zip(mc_models, np.linspace(-0.08, 0.08, len(mc_models)) if len(mc_models) > 1 else [0.0]))
    for model in mc_models:
        c, (pt, reps) = mc_colors[model], mc_test[model]
        for j, k in enumerate(MC_METRICS):
            x0, x1 = j - 0.18 + offs[model], j + 0.18 + offs[model]
            v = mc_valp[model].get(k, np.nan)
            lo, hi = 100 * ci(reps[k])
            if not np.isnan(v):
                ax.plot([x0, x1], [100 * v, 100 * pt[k]], color=c, linewidth=0.9, alpha=0.6)
                ax.plot(x0, 100 * v, mc_markers[model], markerfacecolor="white", markeredgecolor=c,
                        markeredgewidth=1.3, markersize=5.5, zorder=4)
            ax.plot([x1, x1], [lo, hi], color=c, linewidth=1.2, alpha=0.75)
            ax.plot(x1, 100 * pt[k], mc_markers[model], color=c, markersize=5.5, zorder=4)
    ax.set_xticks(range(len(MC_METRICS)), MC_METRICS)
    ax.set_ylim(-3, 103)
    ax.set_ylabel("%")
    ax.grid(axis="x", visible=False)
    for j in range(1, len(MC_METRICS)):
        ax.axvline(j - 0.5, color="0.88", linewidth=0.6)
    panel(ax, "a", "Validation → test")
    names = [STAGE_LONG[s] for s in MC_ORDER]
    for k, model in enumerate(mc_models):
        cax = fig.add_subplot(gs[1 + k])
        r = mc_results[model]
        ok = ~np.isnan(r["P"]).any(axis=1)
        y, pred = r["y"][ok], r["P"][ok].argmax(axis=1)
        counts = np.array([[((y == a) & (pred == b)).sum() for b in range(4)] for a in range(4)])
        rates = counts / np.maximum(counts.sum(axis=1, keepdims=True), 1)
        cax.imshow(rates, cmap="Blues", vmin=0, vmax=1)
        for a in range(4):
            for b in range(4):
                cax.text(b, a, f"{counts[a, b]}", ha="center", va="center", fontsize=7,
                         color="white" if rates[a, b] > 0.6 else "0.1", fontweight="bold" if a == b else "normal")
        cax.grid(False)
        cax.tick_params(length=0)
        cax.set_xticks(range(4), names, fontsize=6.5, rotation=30)
        cax.set_yticks(range(4), names if k == 0 else [""] * 4, fontsize=6.5)
        cax.set_xlabel("predicted", fontsize=7)
        if k == 0:
            cax.set_ylabel("true class", fontsize=7)
        panel(cax, "bcd"[k], f"{lab(model, 'multiclass')}, test")
    fig.legend(handles=[Line2D([], [], color=mc_colors[m], marker=mc_markers[m], linestyle="", markersize=5,
                               label=lab(m, "multiclass")) for m in mc_models]
               + [Line2D([], [], marker="o", markerfacecolor="white", markeredgecolor="0.3", linestyle="",
                         label="(a) hollow: validation (no CI)"),
                  Line2D([], [], marker="o", color="0.3", linestyle="", label="(a) filled: test, bar = 95 % CI"),
                  Patch(facecolor=plt.get_cmap("Blues")(0.75),
                        label="(b) number = slides; colour = share of the true class (row)")],
               loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.0))
    fig.tight_layout(rect=(0, 0.14, 1, 1))
    save(fig, "multiclass", shown=[lab(m, "multiclass") for m in mc_models])
    plt.show()

## 8 · Results tables

Binary: all metrics per center (validation = point estimate, test = 95 % CI). PPV, NPV and F1 depend on the
tumor prevalence (validation 50 %, test 38 %) and must not be compared between the centers.

In [ ]:
def fmt(point, reps, key):
    if reps is None:
        return f"{100 * point[key]:.1f}"
    lo, hi = 100 * ci(reps[key])
    return f"{100 * point[key]:.1f} [{lo:.1f}, {hi:.1f}]"


KEYS = ["AUC", "Sensitivity", "Specificity", "Balanced accuracy", "PPV", "NPV", "F1"]
rows = []
for model in models:
    rows.append({"Model": lab(model), "Center": DOMAIN_LONG["VAL"], **{k: fmt(val_est[model], None, k) for k in KEYS},
                 "Threshold": f"{val_info[model]['threshold']:.3f}"})
    rows.append({"Model": lab(model), "Center": DOMAIN_LONG["TEST"], **{k: fmt(*test_est[model], k) for k in KEYS},
                 "Threshold": f"{results[model]['at_val']['threshold']:.3f}"})
    g, (lo, hi) = gaps[model]
    rows.append({"Model": lab(model), "Center": "Gap (Val − Test)",
                 "AUC": f"{100 * g:+.1f} [{100 * lo:+.1f}, {100 * hi:+.1f}]"})
main_table = pd.DataFrame(rows).set_index(["Model", "Center"]).fillna("") if rows else pd.DataFrame()
main_table.columns.name = "% [95 % CI]"
display(main_table)

rows = []
for model in mc_models:
    rows.append({"Model": lab(model, "multiclass"), "Center": DOMAIN_LONG["VAL"],
                 **{k: (f"{100 * mc_valp[model][k]:.1f}" if k in mc_valp[model] else "–") for k in MC_METRICS}})
    rows.append({"Model": lab(model, "multiclass"), "Center": DOMAIN_LONG["TEST"], **{k: fmt(*mc_test[model], k) for k in MC_METRICS},
                 "κ stored (alphabetical ids, not ordinal)":
                     f"{100 * mc_results[model]['stored']['cohen_kappa_quadratic_weighted']:.1f}"})
mc_table = pd.DataFrame(rows).set_index(["Model", "Center"]).fillna("") if rows else pd.DataFrame()
mc_table

In [ ]:
# Paired comparisons on the test center (E2 encoder, E4 graph augmentation), Holm-corrected.
def holm(pvals):
    p = np.asarray(pvals, float)
    adj, running = np.empty_like(p), 0.0
    for rank, i in enumerate(np.argsort(p)):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj


def boot_p(reps):
    reps = np.asarray(reps, float)
    reps = reps[~np.isnan(reps)]
    return min(1.0, 2 * min((reps <= 0).mean(), (reps >= 0).mean()))


def fmt_p(p):
    return f"< {1 / N_BOOT:.4f}".rstrip("0") if p < 1 / N_BOOT else f"{p:.3f}"


rows, skipped = [], []
for name, model, ref in COMPARISONS:
    if model not in models or ref not in models:
        skipped.append(f"{name}: skipped ({', '.join(m for m in (model, ref) if m not in models)} not usable yet)")
        continue
    p, q = results[model]["prob"], results[ref]["prob"]
    common = ~np.isnan(p) & ~np.isnan(q)
    reps = np.asarray([roc_auc_score(y_test[i], p[i]) - roc_auc_score(y_test[i], q[i])
                       for i in (idx[common[idx]] for idx in boot) if 0 < y_test[i].sum() < len(i)])
    d = roc_auc_score(y_test[common], p[common]) - roc_auc_score(y_test[common], q[common])
    lo, hi = np.percentile(reps, [2.5, 97.5])
    rows.append({"Comparison": name, "Contrast": f"{lab(model)} − {lab(ref)}", "Metric": "slide AUC (test)",
                 "Δ (pp)": f"{100 * d:+.1f}", "95 % CI": f"[{100 * lo:+.1f}, {100 * hi:+.1f}]", "p": boot_p(reps)})
    if (model, "region") in loc and (ref, "region") in loc:
        dm, dr = loc[(model, "region")], loc[(ref, "region")]
        if len(dm["y"]) == len(dr["y"]) and (dm["y"] == dr["y"]).all() and (dm["slide"] == dr["slide"]).all():
            pat = np.array([slide_patient[s] for s in dm["slide"]])
            by_p = {pt: np.flatnonzero(pat == pt) for pt in patients}
            for metric, fn in (("region AP", average_precision_score), ("region AUC", roc_auc_score)):
                rr = []
                for b in range(N_BOOT):
                    idx = np.concatenate([by_p[patients[k]] for k in
                                          np.random.default_rng(SEED + b).integers(0, len(patients), len(patients))])
                    if 0 < dm["y"][idx].sum() < len(idx):
                        rr.append(fn(dm["y"][idx], dm["p"][idx]) - fn(dr["y"][idx], dr["p"][idx]))
                lo_r, hi_r = np.percentile(rr, [2.5, 97.5])
                rows.append({"Comparison": name, "Contrast": f"{lab(model)} − {lab(ref)}", "Metric": metric,
                             "Δ (pp)": f"{100 * (fn(dm['y'], dm['p']) - fn(dr['y'], dr['p'])):+.1f}",
                             "95 % CI": f"[{100 * lo_r:+.1f}, {100 * hi_r:+.1f}]", "p": boot_p(rr)})
        else:
            print(f"{name}: region sets differ between the runs, no paired region comparison")
    greps = (val_est[model]["AUC"] - test_est[model][1]["AUC"]) - (val_est[ref]["AUC"] - test_est[ref][1]["AUC"])
    lo, hi = ci(greps)
    rows.append({"Comparison": name, "Contrast": f"{lab(model)} − {lab(ref)}", "Metric": "slide Gap (Val − Test)",
                 "Δ (pp)": f"{100 * (gaps[model][0] - gaps[ref][0]):+.1f}",
                 "95 % CI": f"[{100 * lo:+.1f}, {100 * hi:+.1f}]", "p": boot_p(greps)})
pairwise = pd.DataFrame(rows)
if len(pairwise):
    pairwise["p (Holm)"] = holm(pairwise["p"])
    pairwise["p"] = pairwise["p"].map(fmt_p)
    pairwise["p (Holm)"] = pairwise["p (Holm)"].map(fmt_p)
    pairwise = pairwise.set_index(["Comparison", "Metric"])
print("\n".join(skipped))
for m, r in CAVEATS.items():
    print(f"† {m}: {r}")
pairwise

## 9 · E4 over five test centers (center rotation)

Each CAMELYON17 center is the test center once (`--split-phase test_c<t>`): validation = the preceding center,
training = the other three (see `docs/experiments/cross_domain_experiments.md`, Splits). Same config and seed for
every fold; only the centers change. (a) Slide AUC on the validation center (hollow, point estimate) and the test
center (filled, 95 % CI from a patient-level bootstrap of that test center). (b) Gap = AUC(Val) − AUC(Test).
(c) Sensitivity and specificity on the test center at the threshold chosen on that fold's validation center.
(d) E4 effect per test center: Δ test AUC = C-aug − C-noaug, paired on the same slides and resampled patients
(> 0 = graph-level augmentation helps); p-values Holm-corrected over the five test centers (table).
The table below lists every fold (including the selected epoch) and mean ± SD over the five test centers.
All folds follow the same protocol (checkpoint = max. validation slide F2, early stopping patience 10); a fold whose
selected checkpoint is an early epoch is part of the result, not an exception.

In [ ]:
def n_val_graphs(val_csv, corpus_dir):
    # Number of validation slides with a graph in the corpus (the validation metrics refer to these).
    names = set(pd.read_csv(val_csv)["slide_name"])
    files = os.listdir(corpus_dir)
    return sum(any(f == f"{n}.pt" or f.startswith(f"{n}_") for f in files) for n in names)


import os
e4 = {}
for corpus, folds in RUNS_E4.items():
    for t, paths in folds.items():
        t = int(t)
        eval_dir, meta = resolve_path(paths.get("eval")), resolve_path(paths.get("meta"))
        if eval_dir is None:
            continue
        m = yaml.safe_load(open(meta))
        corpus_dir = REPO_ROOT / str(find_key(m, "graph_target_dir"))
        phase = find_key(m, "split_phase")
        fold_dir = SPLITS_DIR / phase if phase else SPLITS_DIR
        if phase and phase != f"test_c{t}":
            raise ValueError(f"{corpus} test center {t}: meta split_phase is {phase}")
        test_cohort = pd.read_csv(fold_dir / "test.csv")
        if set(test_cohort["center"]) != {t}:
            raise ValueError(f"{corpus} test center {t}: split file holds centers {set(test_cohort['center'])}")
        test_cohort["label"] = (test_cohort["tumor_status"] == "tumor").astype(int)
        vt = json.load(open(eval_dir / "validation_thresholds" / "validation_thresholds.json"))
        n_val = n_val_graphs(fold_dir / "val.csv", corpus_dir)
        cm_val = val_counts(vt["slide"], n_val)
        preds = pd.read_csv(_one(eval_dir / "split_test", "*_test_predictions.csv"))[["slide_name", "prob", "label"]]
        merged = test_cohort.merge(preds, on="slide_name", how="left", suffixes=("", "_pred"))
        known = merged["prob"].notna()
        if (merged.loc[known, "label"] != merged.loc[known, "label_pred"]).any():
            raise ValueError(f"Label mismatch: {corpus} test center {t}")
        y, p = merged["label"].to_numpy(), merged["prob"].to_numpy()
        thr = json.load(open(_one(eval_dir / "split_test", "*_test_slide_metrics.json")))["at_validation_threshold"]["threshold"]
        if abs(thr - vt["slide"]["threshold"]) > 1e-9:
            raise ValueError(f"{corpus} test center {t}: test threshold differs from the validation threshold")
        pats, inv = np.unique(merged["patient_id"].to_numpy(), return_inverse=True)
        rows_p = [np.flatnonzero(inv == k) for k in range(len(pats))]
        rng_f = np.random.default_rng(SEED + t)
        bs = [np.concatenate([rows_p[k] for k in rng_f.integers(0, len(pats), len(pats))]) for _ in range(N_BOOT)]
        point = binary_metrics(y, p, thr)
        reps = pd.DataFrame([binary_metrics(y[i], p[i], thr) for i in bs])
        hist = json.load(open(resolve_path(paths["history"]))) if paths.get("history") else None
        e4[(corpus, t)] = {"val": {"AUC": vt["validation_slide_auroc"], **rates_from_counts(cm_val)},
                           "test": point, "reps": reps, "thr": thr, "n_val": n_val, "n_test": int(known.sum()),
                           "val_center": int(pd.read_csv(fold_dir / "val.csv")["center"].iloc[0]),
                           "epoch": (selected_epoch(hist) + 1, len(hist["train_loss"])) if hist else None,
                           "y": y, "p": p, "boot": bs}

e4_corpora = [c for c in RUNS_E4 if any((c, t) in e4 for t in range(5))]
e4_colors = {"C-aug": colors.get("Prov-GigaPath", IEEE_COLORS[0]), "C-noaug": IEEE_COLORS[2]}
e4_markers = {"C-aug": "o", "C-noaug": "^"}


def e4_lab(corpus, t):
    return f"{t}†" if f"{corpus} {t}" in E4_CAVEATS else str(t)


# Paired C-aug vs. C-noaug on the same test center (same split, same seed), wherever both folds exist
pair_rows, pair_val = [], {}
for t in range(5):
    if ("C-aug", t) in e4 and ("C-noaug", t) in e4:
        a, b = e4[("C-aug", t)], e4[("C-noaug", t)]
        if not (a["y"] == b["y"]).all():
            raise ValueError(f"test center {t}: different test cohorts")
        common = ~np.isnan(a["p"]) & ~np.isnan(b["p"])
        y = a["y"]
        rr = np.asarray([roc_auc_score(y[i], a["p"][i]) - roc_auc_score(y[i], b["p"][i])
                         for i in (idx[common[idx]] for idx in a["boot"]) if 0 < y[i].sum() < len(i)])
        d = roc_auc_score(y[common], a["p"][common]) - roc_auc_score(y[common], b["p"][common])
        pair_val[t] = (d, np.percentile(rr, [2.5, 97.5]), boot_p(rr))
pair_holm = dict(zip(pair_val, holm([v[2] for v in pair_val.values()]))) if pair_val else {}
for t, (d, (lo, hi), pv) in pair_val.items():
    pair_rows.append({"Test center": e4_lab("C-noaug", t), "Δ test AUC, C-aug − C-noaug (pp)": f"{100 * d:+.1f}",
                      "95 % CI": f"[{100 * lo:+.1f}, {100 * hi:+.1f}]", "p": fmt_p(pv), "p (Holm)": fmt_p(pair_holm[t])})
for label, keep in (("mean ± SD, all folds", lambda t: True),
                    ("mean ± SD, without † folds", lambda t: not any(f"{c} {t}" in E4_CAVEATS for c in RUNS_E4))):
    ds = [100 * pair_val[t][0] for t in pair_val if keep(t)]
    if len(ds) >= 2 and not (label.endswith("† folds") and not E4_CAVEATS):
        pair_rows.append({"Test center": f"{label} (n = {len(ds)})",
                          "Δ test AUC, C-aug − C-noaug (pp)": f"{np.mean(ds):+.1f} ± {np.std(ds, ddof=1):.1f}"})
e4_pairs = pd.DataFrame(pair_rows).set_index("Test center").fillna("") if pair_rows else pd.DataFrame()

if not e4:
    print("No E4 fold run available.")
else:
    fig, axs = plt.subplots(2, 2, figsize=(W, 4.9))
    (ax_a, ax_b), (ax_c, ax_d) = axs
    offs = dict(zip(e4_corpora, np.linspace(-0.13, 0.13, len(e4_corpora)) if len(e4_corpora) > 1 else [0.0]))
    for corpus in e4_corpora:
        c, mk = e4_colors.get(corpus, "0.3"), e4_markers.get(corpus, "o")
        for t in range(5):
            if (corpus, t) not in e4:
                continue
            r, x = e4[(corpus, t)], t + offs[corpus]
            lo, hi = 100 * ci(r["reps"]["AUC"])
            ax_a.plot([x, x], [lo, hi], color=c, linewidth=1.2, alpha=0.75)
            ax_a.plot(x - 0.07, 100 * r["val"]["AUC"], mk, markerfacecolor="white", markeredgecolor=c,
                      markeredgewidth=1.2, markersize=5, zorder=4)
            ax_a.plot(x, 100 * r["test"]["AUC"], mk, color=c, markersize=6, zorder=4)
            g = r["val"]["AUC"] - r["test"]["AUC"]
            glo, ghi = 100 * ci(r["val"]["AUC"] - r["reps"]["AUC"])
            ax_b.plot([x, x], [glo, ghi], color=c, linewidth=1.2, alpha=0.75)
            ax_b.plot(x, 100 * g, mk, color=c, markersize=6, zorder=4)
            ax_c.plot(x - 0.05, 100 * r["test"]["Sensitivity"], mk, color=c, markersize=6, zorder=4)
            ax_c.plot(x + 0.05, 100 * r["test"]["Specificity"], mk, markerfacecolor="white", markeredgecolor=c,
                      markeredgewidth=1.2, markersize=6, zorder=4)
            ax_c.plot([x - 0.05, x + 0.05], [100 * r["test"]["Sensitivity"], 100 * r["test"]["Specificity"]],
                      color=c, linewidth=0.8, alpha=0.6)
    ticks = [f"{t}\n(val {e4[(e4_corpora[0], t)]['val_center'] if (e4_corpora[0], t) in e4 else (t - 1) % 5})" for t in range(5)]
    caveat_folds = {int(k.split()[-1]) for k in E4_CAVEATS}
    ticks = [f"{t}†" + tk[1:] if t in caveat_folds else tk for t, tk in enumerate(ticks)]
    for ax, letter, title, ylab in ((ax_a, "a", "Slide AUC", "%"), (ax_b, "b", "Gap = Val − Test", "pp"),
                                    (ax_c, "c", "Test sens. / spec.", "%")):
        ax.set_xticks(range(5), ticks, fontsize=6.5)
        ax.set_xlim(-0.5, 4.5)
        ax.set_xlabel("Test center (validation center)", fontsize=7)
        ax.set_ylabel(ylab)
        ax.grid(axis="x", visible=False)
        panel(ax, letter, title)
    ax_b.axhline(0, color="0.3", linestyle=":", linewidth=0.9)
    ax_c.set_ylim(-3, 103)
    for t, (d, (lo, hi), _) in pair_val.items():
        ax_d.plot([t, t], [100 * lo, 100 * hi], color="0.25", linewidth=1.4)
        ax_d.plot(t, 100 * d, "D", color="0.25", markersize=5.5, zorder=4)
        ax_d.annotate(f"{100 * d:+.1f}", (t, 100 * hi), xytext=(0, 3), textcoords="offset points", ha="center",
                      va="bottom", fontsize=6.5, color="0.2")
    ax_d.axhline(0, color="0.3", linestyle=":", linewidth=0.9)
    if pair_val:
        hi_all = max(100 * v[1][1] for v in pair_val.values())
        lo_all = min(100 * v[1][0] for v in pair_val.values())
        ax_d.set_ylim(min(lo_all, 0) - 3, max(hi_all, 0) + 0.15 * (max(hi_all, 0) - min(lo_all, 0)) + 3)
    ax_d.set_xticks(range(5), ticks, fontsize=6.5)
    ax_d.set_xlim(-0.5, 4.5)
    ax_d.set_xlabel("Test center (validation center)", fontsize=7)
    ax_d.set_ylabel("pp")
    ax_d.grid(axis="x", visible=False)
    panel(ax_d, "d", "Δ test AUC, C-aug − C-noaug (paired)")
    handles = [Line2D([], [], color=e4_colors.get(cp, "0.3"), marker=e4_markers.get(cp, "o"), linestyle="", markersize=5,
                      label=f"{cp}" + ("" if all((cp, t) in e4 for t in range(5)) else
                                       f" (only test center{'s' if sum((cp, t) in e4 for t in range(5)) > 1 else ''} {', '.join(str(t) for t in range(5) if (cp, t) in e4)})"))
               for cp in e4_corpora]
    handles += [Line2D([], [], marker="o", markerfacecolor="white", markeredgecolor="0.3", linestyle="",
                       label="(a) hollow: validation (no CI); filled: test"),
                Line2D([], [], color="0.4", marker="|", markersize=10, markeredgewidth=1.2, linestyle="",
                       label="(a, b) bar: 95 % CI (test-center sampling)"),
                Line2D([], [], marker="o", color="0.3", linestyle="", label="(c) filled: sensitivity, at the val. threshold"),
                Line2D([], [], marker="o", markerfacecolor="white", markeredgecolor="0.3", linestyle="",
                       label="(c) hollow: specificity"),
                Line2D([], [], marker="D", color="0.25", linestyle="-", markersize=5,
                       label="(d) Δ with paired 95 % CI (> 0: augmentation better)")]
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.0))
    fig.tight_layout(rect=(0, 0.11, 1, 1))
    import textwrap
    notes = [f"† {k}: {v}" for k, v in E4_CAVEATS.items()]
    if notes:
        fig.text(0.01, -0.01, "\n".join(textwrap.fill(n, 190) for n in notes), fontsize=6, color="0.3",
                 ha="left", va="top")
    if SAVE_PREFIX:
        for ext in ("png", "pdf"):
            fig.savefig(OUT_DIR / f"{SAVE_PREFIX}_e4_folds.{ext}", bbox_inches="tight")
        print("Saved:", OUT_DIR / f"{SAVE_PREFIX}_e4_folds.png")
    plt.show()

rows = []
for corpus in e4_corpora:
    for t in range(5):
        if (corpus, t) not in e4:
            continue
        r = e4[(corpus, t)]
        lo, hi = 100 * ci(r["reps"]["AUC"])
        rows.append({"Corpus": corpus, "Test center": e4_lab(corpus, t), "Val center": r["val_center"],
                     "Val AUC": f"{100 * r['val']['AUC']:.1f}", "Test AUC": f"{100 * r['test']['AUC']:.1f} [{lo:.1f}, {hi:.1f}]",
                     "Gap": f"{100 * (r['val']['AUC'] - r['test']['AUC']):+.1f}",
                     "Test sens.": f"{100 * r['test']['Sensitivity']:.1f}", "Test spec.": f"{100 * r['test']['Specificity']:.1f}",
                     "Threshold": f"{r['thr']:.3f}", "Epoch (sel./run)": "{}/{}".format(*r["epoch"]) if r["epoch"] else "–",
                     "Slides val / test": f"{r['n_val']} / {r['n_test']}"})
    for label, keep in (("mean ± SD, all folds", lambda t: True),
                        ("mean ± SD, without † folds", lambda t: not any(f"{c} {t}" in E4_CAVEATS for c in RUNS_E4))):
        ts = [t for t in range(5) if (corpus, t) in e4 and keep(t)]
        if len(ts) < 2 or (label.endswith("† folds") and not E4_CAVEATS):
            continue
        def ms(f):
            v = 100 * np.array([f(e4[(corpus, t)]) for t in ts])
            return f"{v.mean():.1f} ± {v.std(ddof=1):.1f}"
        rows.append({"Corpus": corpus, "Test center": f"{label} (n = {len(ts)})", "Val center": "",
                     "Val AUC": ms(lambda r: r["val"]["AUC"]), "Test AUC": ms(lambda r: r["test"]["AUC"]),
                     "Gap": ms(lambda r: r["val"]["AUC"] - r["test"]["AUC"]),
                     "Test sens.": ms(lambda r: r["test"]["Sensitivity"]), "Test spec.": ms(lambda r: r["test"]["Specificity"])})
e4_table = pd.DataFrame(rows).set_index(["Corpus", "Test center"]).fillna("") if rows else pd.DataFrame()
e4_table.columns.name = "% [95 % CI]"

missing = [f"{c} test center {t}" for c in RUNS_E4 for t in range(5) if (c, t) not in e4]
print("Not trained / not evaluated:", ", ".join(missing) if missing else "none")
for k, v in E4_CAVEATS.items():
    print(f"† {k}: {v}")
display(e4_table)
e4_pairs

## 10 · Key results by experiment (generated)

Generated from the numbers above; † marks a run with a caveat. A comparison is only stated when both runs exist.

In [ ]:
def slide_line(model):
    pt, reps = test_est[model]
    lo, hi = 100 * ci(reps["AUC"])
    g, (glo, ghi) = gaps[model]
    return (f"{lab(model)}: val AUC {100 * val_est[model]['AUC']:.1f}, test AUC {100 * pt['AUC']:.1f} [{lo:.1f}, {hi:.1f}], "
            f"Gap {100 * g:+.1f} [{100 * glo:+.1f}, {100 * ghi:+.1f}] "
            f"({'CI excludes 0' if glo > 0 or ghi < 0 else 'CI includes 0'}); test sens. {100 * pt['Sensitivity']:.1f} / "
            f"spec. {100 * pt['Specificity']:.1f} at the val. threshold (oracle bal. acc. +{100 * (oracle[model][0] - pt['Balanced accuracy']):.1f} pp)")


def loc_line(model):
    parts = []
    for level in LEVELS:
        if (model, level) in loc_est:
            point = loc_est[(model, level)][0]
            prev = loc[(model, level)]["y"].mean()
            parts.append(f"{level} AP {100 * point['AP']:.1f} (random {100 * prev:.1f}), AUC {100 * point['AUC']:.1f}")
    return f"{lab(model)} localization: " + "; ".join(parts) if parts else None


def comparison_lines(name):
    if not len(pairwise) or name not in pairwise.index.get_level_values(0):
        return [f"  comparison not possible yet ({next(s for s in skipped if s.startswith(name)) if any(s.startswith(name) for s in skipped) else 'missing run'})"]
    return [f"  {metric}: {r['Contrast']} {r['Δ (pp)']} pp {r['95 % CI']}, p (Holm) {r['p (Holm)']}"
            for (n, metric), r in pairwise.iterrows() if n == name]


sections = {
    "E2 binary (encoder)": ["Prov-GigaPath", "ResNet-50"],
    "E4 (graph-level augmentation)": ["Prov-GigaPath", "Prov-GigaPath · C-noaug"],
}
comparison_of = {"E2 binary (encoder)": "E2 encoder", "E4 (graph-level augmentation)": "E4 graph augmentation"}
for title, runs in sections.items():
    print(title)
    for m in runs:
        if m in models:
            print("  " + slide_line(m))
            if loc_line(m):
                print("  " + loc_line(m))
        else:
            print(f"  {m}: not available yet")
    print("\n".join(comparison_lines(comparison_of[title])))
    print()
print("E2 multiclass (encoder)")
for m in RUNS_MC:
    if m in mc_models:
        pt, reps = mc_test[m]
        klo, khi = 100 * ci(reps["Quadratic κ"])
        ok = ~np.isnan(mc_results[m]["P"]).any(axis=1)
        y, pred = mc_results[m]["y"][ok], mc_results[m]["P"][ok].argmax(axis=1)
        recall = ", ".join(f"{STAGE_LONG[s]} {100 * (pred[y == k] == k).mean():.0f} % ({int((y == k).sum())})"
                           for k, s in enumerate(MC_ORDER))
        print(f"  {lab(m, 'multiclass')}: test κ_quad {100 * pt['Quadratic κ']:.1f} [{klo:.1f}, {khi:.1f}] "
              f"(val {100 * mc_valp[m].get('Quadratic κ', np.nan):.1f}), macro-F1 {100 * pt['Macro-F1']:.1f}, "
              f"AUROC {100 * pt['AUROC (OvO)']:.1f}; per-class recall: {recall}")
    else:
        print(f"  {m}: not available yet")
print("\nE4 over five test centers")
for corpus in e4_corpora:
    for t in range(5):
        if (corpus, t) in e4:
            r = e4[(corpus, t)]
            lo, hi = 100 * ci(r["reps"]["AUC"])
            print(f"  {corpus} test center {e4_lab(corpus, t)}: val AUC {100 * r['val']['AUC']:.1f}, test AUC "
                  f"{100 * r['test']['AUC']:.1f} [{lo:.1f}, {hi:.1f}], Gap {100 * (r['val']['AUC'] - r['test']['AUC']):+.1f}")
for (corpus, label), row in e4_table.iterrows():
    if label.startswith("mean"):
        print(f"  {corpus} {label}: test AUC {row['Test AUC']}, Gap {row['Gap']}")
for t, row in e4_pairs.iterrows():
    if str(t).startswith("mean"):
        print(f"  C-aug − C-noaug, {t}: {row['Δ test AUC, C-aug − C-noaug (pp)']} pp")
    else:
        print(f"  test center {t}: C-aug − C-noaug {row['Δ test AUC, C-aug − C-noaug (pp)']} pp {row['95 % CI']}, p (Holm) {row['p (Holm)']}")
print("  missing:", ", ".join(missing) if missing else "none")
for k, v in E4_CAVEATS.items():
    print(f"  † {k}: {v}")
for m, r in CAVEATS.items():
    print(f"\n† {m}: {r}")

In [ ]:
if SAVE_PREFIX:
    for name, table in (("cohort", cohort_table), ("main", main_table), ("size", size_table),
                        ("localization", loc_table), ("multiclass", mc_table), ("pairwise", pairwise),
                        ("e4_folds", e4_table), ("e4_pairs", e4_pairs)):
        if len(table):
            table.to_csv(OUT_DIR / f"{SAVE_PREFIX}_table_{name}.csv")
            table.to_latex(OUT_DIR / f"{SAVE_PREFIX}_table_{name}.tex", escape=True)
    print("Tables written to", OUT_DIR)